タイタニック号の生存者を回帰モデルを用いて予測するプログラム

ランダムフォレスト回帰

決定木回帰

GDBT

サポートベクター回帰

がそれぞれ使える。

In [1]:
import os

# ローカルパスの設定
raw_dir     = '../data/raw/'
split_dir   = '../data/split_from_train/'
missing_dir = '../data/missing_store/'

# 出力ディレクトリの作成
os.makedirs(split_dir,   exist_ok=True)
os.makedirs(missing_dir, exist_ok=True)

print("パス設定完了")


パス設定完了


titanic dataを読み込む

- `data/raw/train.csv` ：学習・評価に使うデータ（Survived ラベル付き）
- `data/raw/test.csv` ：評価には使わない。欠損値補完の参照用として読み込み、`data/missing_store/` に保管する


In [2]:
import numpy as np
import pandas as pd

#
# 学習データの読み込み (data/raw/train.csv)
#
df = pd.read_csv(raw_dir + 'train.csv')

#
# テストデータの読み込み。
# READMEの方針: 評価には使わず、欠損値補完の参照用として data/missing_store/ に保管する
#
test_df = pd.read_csv(raw_dir + 'test.csv')
test_df.to_csv(missing_dir + 'test.csv', index=False)

print(f"学習データ: {df.shape}")
print(f"参照用テストデータ: {test_df.shape}")


学習データ: (891, 12)
参照用テストデータ: (418, 11)


dataの欠損値を補完する必要があります。

titanic dataには欠損値がたくさんあるので、何らかの補完をしておかないと学習ができないから。


In [3]:
print("titanic data の欠損値")

print(df.isnull().sum())

titanic data の欠損値
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [4]:
#
# 年齢の平均値と中央値を確認するときにはコメント外してちょんまげ
#
# print(df.Age.mean())
# print(df.Age.median())

# 学習データとテストデータを連結する
all_df = pd.concat([df, test_df], ignore_index=True)
# print(fill_df)

#
# 学習データとテストデータが連結されていることを確認。
# 中央値は全体の値から計算したいのでこの手法をとる
#
print(all_df.shape)

#
# 念の為、dfをfill_dfにコピー
#
fill_df = df.copy()

#
# 年齢の中央値を全てのデータから計算
#
age_median=all_df.Age.median()

#
# 年齢の欠損値を、計算しておいた中央値で補完する
# ここの年齢補完、家族情報（family name, SibSp, Parch）を使う手段もあるので、よくデータを眺めてみてください。
#
fill_df.Age = fill_df.Age.fillna(age_median)

#
# Embarked 欠損値の補完：どちらもS（サザンプトン港）としてしまう。
# ここも同じファミリーは同じ場所から乗り込んだと考えるのが妥当。よく調べてからの方が良いと思います。
#
fill_df.Embarked = fill_df.Embarked.fillna('S')

#
# 乗船した港の欠損値を再度確認する
#
print("欠損値の確認")
print(fill_df.isnull().sum())
# fill_df.head()

(1309, 12)
欠損値の確認
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

#
# y_all に Survived を挿入する
#
y_all = fill_df["Survived"]

#
# 使わないカラムを除く（Survived, Name, Ticket, Cabin）
#
x_all = fill_df.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin'])

#
# Sex, Embarkedをone-hotエンコードに変換する
#
x_all = pd.get_dummies(x_all, columns=['Sex', 'Embarked'], prefix='HE')

#
# 8:2 に分割する（random_state で再現性を確保）
#
x_train, x_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.2, random_state=42)

#
# 分割したデータを data/split_from_train/ に保存する（READMEの方針）
#
train_split_df = x_train.copy()
train_split_df.insert(0, 'Survived', y_train.values)
train_split_df.to_csv(split_dir + 'train_split.csv', index=False)

valid_split_df = x_test.copy()
valid_split_df.insert(0, 'Survived', y_test.values)
valid_split_df.to_csv(split_dir + 'valid_split.csv', index=False)

print(f"学習用: {x_train.shape}, 評価用: {x_test.shape}")
print(f"分割データを {split_dir} に保存しました")
print(x_train.head())


学習用: (712, 11), 評価用: (179, 11)
分割データを ../data/split_from_train/ に保存しました
     PassengerId  Pclass   Age  SibSp  Parch     Fare  HE_female  HE_male  \
331          332       1  45.5      0      0  28.5000      False     True   
733          734       2  23.0      0      0  13.0000      False     True   
382          383       3  32.0      0      0   7.9250      False     True   
704          705       3  26.0      1      0   7.8542      False     True   
813          814       3   6.0      4      2  31.2750       True    False   

      HE_C   HE_Q  HE_S  
331  False  False  True  
733  False  False  True  
382  False  False  True  
704  False  False  True  
813  False  False  True  


ランダムフォレスト回帰

In [6]:
# ライブラリのインポート
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score
#
# ランダムフォレスト回帰モデルを作る
#
model = RandomForestRegressor()

#
# ランダムフォレスト回帰モデルを学習
#
model.fit(x_train, y_train)

#
# 作成したランダムフォレスト回帰モデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[0. 0. 0. 1. 0. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 1. 0. 0.
 0. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1. 1. 0. 1. 1. 0. 0. 1. 0. 0. 0. 1. 1. 1.
 1. 1. 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0.
 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0.
 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1.
 0. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]
推定率= 81.01 [%]


RMSE : Root Mean Squre Errorの計算

In [7]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))



学習データのRMSE(train)=0.1437
テストデータのRMSE(train)=0.3690


決定木回帰

In [8]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor()
model.fit(x_train, y_train)

#
# 作成した決定木回帰モデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[0. 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 1. 0. 0.
 0. 1. 0. 1. 1. 1. 0. 0. 0. 1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0.
 0. 1. 1. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 1. 0. 1. 0. 1. 1. 1.
 0. 1. 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0.
 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 1. 0. 0.
 0. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1.
 0. 0. 0. 1. 0. 0. 0. 0. 0. 1. 1.]
推定率= 75.98 [%]


In [9]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))


学習データのRMSE(train)=0.0000
テストデータのRMSE(train)=0.4901


GBDT

In [10]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor()

model.fit(x_train, y_train)

#
# 作成したGDBTモデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[0. 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 1. 1. 0. 0. 1. 0. 0. 0. 1. 1. 1.
 0. 1. 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0.
 0. 1. 1. 1. 0. 0. 1. 0. 1. 0. 1. 0. 0. 1. 1. 1. 1. 0. 0. 0. 0. 1. 0. 0.
 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1.
 0. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0.]
推定率= 82.12 [%]


In [11]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))


学習データのRMSE(train)=0.2833
テストデータのRMSE(train)=0.3665


ガウスカーネルのサポートベクター回帰

In [12]:
from sklearn.svm import SVR

model = SVR()

model.fit(x_train, y_train)

#
# 作成したGDBTモデルを使った予測をおこなう。
#
y_pred = model.predict(x_test)

#
# 予測値を0.5未満は0, 0.5以上は1に変換する
#
y_pred_bin = np.zeros(len(y_pred))
for i in range(len(y_pred)):
  if y_pred[i] < 0.5:
    y_pred_bin[i] = 0
  else:
    y_pred_bin[i] = 1
#
# 予測結果を表示
#
print(y_pred_bin)

#
# 予測結果y_predと正しい結果y_testを比較して、認識率を計算する
#
accuracy = accuracy_score(y_test, y_pred_bin)

#
# 予測値を表示する
#
acc = accuracy*100
# print('推定率= %.2f' % acc)
print('推定率= %.2f' % (accuracy*100), "[%]")

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
推定率= 62.01 [%]


In [13]:
from sklearn.metrics import mean_squared_error

y_train_pred = model.predict(x_train)
print('学習データのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_train, y_train_pred))))

y_test_pred = model.predict(x_test)
print('テストデータのRMSE(train)=%.4f' % (np.sqrt(mean_squared_error(y_test, y_test_pred))))

学習データのRMSE(train)=0.4998
テストデータのRMSE(train)=0.5158
